# 03 Evaluation

Runs the eval set, computes retrieval hit rate at top 5 and an LLM-as-judge correctness score.
Logs everything to MLflow.

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_PATH = "/Workspace/Repos/nyaya-bodh"
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

os.environ.setdefault("ARTIFACTS_DIR", "/dbfs/FileStore/nyaya_bodh")
try:
except Exception:
    pass

In [ ]:
import mlflow
from config import CONFIG
from src.llm import chat
from src.pipeline import NyayaBodhPipeline
from src.prompts import JUDGE_SYSTEM_PROMPT, build_judge_prompt

EVAL_PATH = Path(REPO_PATH) / "data" / "eval_questions.json"
eval_set = json.loads(EVAL_PATH.read_text())["items"]
print(f"Loaded {len(eval_set)} eval items")

In [ ]:
pipeline = NyayaBodhPipeline()

def judge(question: str, expected: str, answer: str) -> dict:
    raw = chat(JUDGE_SYSTEM_PROMPT, build_judge_prompt(question, expected, answer))
    raw = raw.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"correctness": 0, "citation": 0, "reason": "judge parse failure"}

In [ ]:
mlflow.set_experiment(CONFIG.mlflow_experiment_name)
results = []
hit_at_5 = 0
hit_at_1 = 0
total_correctness = 0
total_citation = 0

with mlflow.start_run(run_name="eval"):
    for item in eval_set:
        response = pipeline.answer(item["question"], include_schemes=False)
        retrieved_ids = [c.section_id for c in response.citations]
        in_top_5 = item["expected_section"] in retrieved_ids
        in_top_1 = bool(retrieved_ids) and retrieved_ids[0] == item["expected_section"]
        verdict = judge(item["question"], item["expected_section"], response.answer)

        hit_at_5 += int(in_top_5)
        hit_at_1 += int(in_top_1)
        total_correctness += int(verdict.get("correctness", 0))
        total_citation += int(verdict.get("citation", 0))

        results.append(
            {
                "question": item["question"],
                "expected_section": item["expected_section"],
                "language": item["language"],
                "retrieved": retrieved_ids,
                "in_top_5": in_top_5,
                "in_top_1": in_top_1,
                "answer": response.answer,
                "judge": verdict,
            }
        )

    n = len(eval_set)
    metrics = {
        "hit_at_5": hit_at_5 / n,
        "hit_at_1": hit_at_1 / n,
        "judge_correctness_avg": total_correctness / (n * 5),
        "judge_citation_avg": total_citation / n,
        "n_examples": n,
    }
    mlflow.log_metrics(metrics)
    mlflow.log_dict({"results": results}, "eval_results.json")

print(json.dumps(metrics, indent=2))